# LLM 입력을 위한 최종 프롬프트 생성 노트북

이 노트북은 전처리된 모든 데이터(제품, 페르소나, 경쟁사)를 불러와, 제품과 페르소나의 모든 조합에 대한 프롬프트를 생성하고 `prompts_for_llm.jsonl` 파일로 저장합니다.

## 1. 설정 및 라이브러리 임포트

In [5]:
import json
import pandas as pd

# --- 파일 경로 설정 ---
# 입력 파일
PRODUCT_FILE = 'product_info_preprocessed.jsonl' # '동원' 브랜드가 추가된 제품 파일
PERSONA_FILE = 'persona_attributes_weighted.jsonl'
COMPETITOR_FILE_CLEANED = 'competitor_prices_cleaned_final.csv' # 최종 전처리된 경쟁사 데이터

# 최종 출력 파일
OUTPUT_FILE = 'prompts_for_llm.jsonl'

## 2. 경쟁사 시장 데이터 불러오기

In [6]:
def load_market_context(filepath):
    """
    전처리된 경쟁사 가격 CSV 파일을 읽어, 카테고리별 가격 범위를 담은 딕셔너리를 생성합니다.
    """
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"오류: '{filepath}' 파일을 찾을 수 없습니다.")
        return None

    market_context = {}
    if 'category' in df.columns:
        for category, group in df.groupby('category'):
            min_price = group['price_per_100g'].min()
            max_price = group['price_per_100g'].max()
            market_context[category] = {
                "competitor_price_range_per_100g": f"{int(min_price):,}원 ~ {int(max_price):,}원"
            }
    return market_context

print("1. 전처리된 경쟁사 가격 데이터를 불러옵니다...")
market_context = load_market_context(COMPETITOR_FILE_CLEANED)
if market_context is not None:
    main_categories_from_market = list(market_context.keys())
    print("   => 완료. 카테고리별 시장 데이터가 준비되었습니다.")
    print(f"   (감지된 시장 카테고리: {main_categories_from_market})")

1. 전처리된 경쟁사 가격 데이터를 불러옵니다...
   => 완료. 카테고리별 시장 데이터가 준비되었습니다.
   (감지된 시장 카테고리: ['RTD_액상커피', '그릭요거트', '참치액', '참치캔', '캔햄'])


## 3. 제품 및 페르소나 데이터 불러오기

In [7]:
print(f"2. '{PRODUCT_FILE}'와 '{PERSONA_FILE}' 파일을 불러옵니다...")
with open(PRODUCT_FILE, 'r', encoding='utf-8') as f:
    products = [json.loads(line) for line in f]
with open(PERSONA_FILE, 'r', encoding='utf-8') as f:
    personas = [json.loads(line) for line in f]
print(f"   => 완료. 제품 {len(products)}개, 페르소나 {len(personas)}개를 불러왔습니다.")

2. 'product_info_preprocessed.jsonl'와 'persona_attributes_weighted.jsonl' 파일을 불러옵니다...
   => 완료. 제품 15개, 페르소나 363개를 불러왔습니다.


## 4. 프롬프트 생성 함수 정의

In [8]:
def map_to_main_category(detailed_category, main_categories):
    """
    제품의 상세 카테고리 문자열을 보고, 5대 대표 카테고리 중 어디에 속하는지 찾아줍니다.
    """
    for main_cat in main_categories:
        if main_cat in detailed_category:
            return main_cat
    if '커피' in detailed_category: # RTD_액상커피 예외 처리
        return 'RTD_액상커피'
    return None

def create_single_prompt(product_info, persona_info, market_context, main_categories):
    """
    제품, 페르소나, 시장 정보를 바탕으로 최종 프롬프트를 생성합니다.
    """
    product_str = json.dumps(product_info, ensure_ascii=False, indent=4)
    persona_str = json.dumps(persona_info, ensure_ascii=False, indent=4)

    detailed_category = product_info.get('category', '')
    main_category = map_to_main_category(detailed_category, main_categories)
    context_data = market_context.get(main_category, {})
    context_str = json.dumps(context_data, ensure_ascii=False, indent=4) if context_data else "{}"

    prompt_template = f"""# ROLE
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 \"제품 정보\", 원시 \"페르소나 데이터\", 그리고 \"시장 경쟁 환경\"을 종합적으로 분석하여, 페르소나가 해당 제품의 잠재 구매자로서 어떤 특징을 보일지 예측하고, 그 결과를 하나의 완결된 JSON 객체로 생성하는 것입니다.

# INSTRUCTION
아래의 모든 정보를 바탕으로, 페르소나가 해당 제품의 구매자로서 성립하는 **싱글턴 페르소나 JSON**을 생성하세요. 페르소나의 속성(attributes)과 제품의 특징(features)을 논리적으로 연결하여 구매 확률과 이유, 월별 구매 빈도를 예측해야 합니다.

# INPUT DATA
## 1. 제품 정보
{product_str}

## 2. 페르소나 데이터
{persona_str}

## 3. 시장 경쟁 환경
{context_str}

# OUTPUT FORMAT
반드시 아래와 같은 구조의 JSON 형식으로만 응답하세요. 다른 설명은 추가하지 마세요.

{{{{
  \"product_name\": \"{product_info.get('product_name', '')}\",
  \"persona_key\": {persona_info.get('persona_key')},
  \"purchase_behavior_prediction\": {{{{
    \"purchase_probability_pct\": \"<여기에 구매 확률(0-100)을 숫자로 예측>\",
    \"reason\": \"<여기에 페르소나 속성과 제품 특징, 시장 상황을 연결한 구매 결정 이유를 상세히 서술>\",
    \"monthly_purchase_frequency\": {{{{
      \"2024-07\": 0, \"2024-08\": 0, \"2024-09\": 0, \"2024-10\": 0, \"2024-11\": 0, \"2024-12\": 0,
      \"2025-01\": 0, \"2025-02\": 0, \"2025-03\": 0, \"2025-04\": 0, \"2025-05\": 0, \"2025-06\": 0
    }}}}
  }}}}
}}}} """
    return prompt_template.strip()

print("프롬프트 생성 함수가 정의되었습니다.")

프롬프트 생성 함수가 정의되었습니다.


## 5. 모든 조합에 대한 프롬프트 생성 및 저장

In [ ]:
if 'market_context' in locals() and market_context is not None:
    print("3. 모든 조합에 대한 프롬프트 생성을 시작합니다...")
    all_prompts = []
    total_combinations = len(products) * len(personas)

    for i, product in enumerate(products):
        for j, persona in enumerate(personas):
            product_essentials = {
                "brand": product.get("brand", "동원 F&B"),
                "product_name": product.get("product_name"),
                "category": product.get("category"),
                "features": product.get("features"),
                "targeted_consumer": product.get("targeted_consumer"),
                "price_text": product.get("price_text"),
                "advertise_info": product.get("advertise_info")
            }
            prompt = create_single_prompt(product_essentials, persona, market_context, main_categories_from_market)
            all_prompts.append({
                "product_name": product.get("product_name"),
                "persona_key": persona.get("persona_key"),
                "prompt": prompt
            })
        print(f"  - {i+1}번째 제품 '{product.get('product_name')}'에 대한 {len(personas)}개 페르소나 프롬프트 생성 완료")

    print(f"\n4. 생성된 프롬프트를 '{OUTPUT_FILE}' 파일로 저장합니다...")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in all_prompts:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("="*40)
    print("🎉 모든 작업이 성공적으로 완료되었습니다!")
    print(f"총 {len(all_prompts)}개의 프롬프트가 생성되어 '{OUTPUT_FILE}'에 저장되었습니다.")
else:
    print("시장 데이터(market_context)가 로드되지 않아 프롬프트 생성을 건너뜁니다.")

3. 모든 조합에 대한 프롬프트 생성을 시작합니다...
  - 1번째 제품 '덴마크 하이그릭요거트 400g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 2번째 제품 '동원맛참 고소참기름 135g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 3번째 제품 '동원맛참 고소참기름 90g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 4번째 제품 '동원맛참 매콤참기름 135g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 5번째 제품 '동원맛참 매콤참기름 90g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 6번째 제품 '동원참치액 순 500g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 7번째 제품 '동원참치액 순 900g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 8번째 제품 '동원참치액 진 500g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 9번째 제품 '동원참치액 진 900g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 10번째 제품 '리챔 오믈레햄 200g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 11번째 제품 '리챔 오믈레햄 340g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 12번째 제품 '소화가 잘되는 우유로 만든 바닐라라떼 250mL'에 대한 363개 페르소나 프롬프트 생성 완료
  - 13번째 제품 '소화가 잘되는 우유로 만든 카페라떼 250mL'에 대한 363개 페르소나 프롬프트 생성 완료
  - 14번째 제품 '프리미엄 동원참치액 500g'에 대한 363개 페르소나 프롬프트 생성 완료
  - 15번째 제품 '프리미엄 동원참치액 900g'에 대한 363개 페르소나 프롬프트 생성 완료

4. 생성된 프롬프트를 'prompts_for_llm.jsonl' 파일로 저장합니다...
🎉 모든 작업이 성공적으로 완료되었습니다!
총 5445개의 프롬프트가 생성되어 'prompts_for_llm.jsonl'에 저장되었습니다.

--- 생성된 prompts_for_l

In [11]:
# --- 추가/수정된 코드: 결과 미리보기 및 파일 저장 ---
print("\n--- 생성된 prompts_for_llm.jsonl 파일 미리보기 (상위 2개) ---")

preview_prompts = []
preview_filename = "prompts_preview.json"

try:
    # 1. 원본 파일에서 상위 2개의 프롬프트를 읽어옵니다.
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < 2:
                prompt_object = json.loads(line)
                preview_prompts.append(prompt_object)
                
                # 화면에도 예쁘게 출력합니다.
                print(f"\n--- 프롬프트 #{i+1} ---")
                print(json.dumps(prompt_object, ensure_ascii=False, indent=2))
            else:
                break
    
    # 2. 읽어온 2개의 프롬프트를 별도의 JSON 파일로 저장합니다.
    if preview_prompts:
        with open(preview_filename, 'w', encoding='utf-8') as f_out:
            # json.dump를 사용하여 리스트 전체를 하나의 JSON 파일로 저장합니다.
            json.dump(preview_prompts, f_out, ensure_ascii=False, indent=2)
        print(f"\n✅ 미리보기 파일 '{preview_filename}'이 성공적으로 저장되었습니다.")

except FileNotFoundError:
    print(f"'{OUTPUT_FILE}'을 찾을 수 없어 미리보기를 표시하거나 저장할 수 없습니다.")
except Exception as e:
    print(f"미리보기를 처리하는 중 오류가 발생했습니다: {e}")


--- 생성된 prompts_for_llm.jsonl 파일 미리보기 (상위 2개) ---

--- 프롬프트 #1 ---
{
  "product_name": "덴마크 하이그릭요거트 400g",
  "persona_key": 1,
  "prompt": "# ROLE\n당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 \"제품 정보\", 원시 \"페르소나 데이터\", 그리고 \"시장 경쟁 환경\"을 종합적으로 분석하여, 페르소나가 해당 제품의 잠재 구매자로서 어떤 특징을 보일지 예측하고, 그 결과를 하나의 완결된 JSON 객체로 생성하는 것입니다.\n\n# INSTRUCTION\n아래의 모든 정보를 바탕으로, 페르소나가 해당 제품의 구매자로서 성립하는 **싱글턴 페르소나 JSON**을 생성하세요. 페르소나의 속성(attributes)과 제품의 특징(features)을 논리적으로 연결하여 구매 확률과 이유, 월별 구매 빈도를 예측해야 합니다.\n\n# INPUT DATA\n## 1. 제품 정보\n{\n    \"brand\": \"동원 F&B\",\n    \"product_name\": \"덴마크 하이그릭요거트 400g\",\n    \"category\": \"우유류 > 발효유 > 호상-중대용량\",\n    \"features\": [\n        \"건강식품\",\n        \"고단백\",\n        \"고소한맛\",\n        \"높은 만족도\"\n    ],\n    \"targeted_consumer\": [\n        \"유당불내증\"\n    ],\n    \"price_text\": \"3,980원\",\n    \"advertise_info\": \"광고/프로모션: 2025년 6-7월, 일반인 광고\"\n}\n\n## 2. 페르소나 데이터\n{\n    \"persona_key\": 1,\n    \"attributes\": {\n        \"gender\": {\n           